# 松尾研 Physical AI｜工程88・実写17枚・畑の粗選果（既知画像教師あり）
作成：2026/10/30 07:06 JST

**以前のSTEMFIX 10/10は保全。本版のPyBullet 10/10はまだ未検証。** 元写真17枚・上面/下面合成UVを継承。B/Cを含む既知9個のカメラ画像から教師あり粗選果モデルを再学習し、3本指カメラ画像で分類する。モデルには結果ケース番号・等級正解を実行時に渡さない。未知りんごの精度は保証しない。


In [ ]:
from pathlib import Path
import subprocess
fonts=[Path('/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'),Path('/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc')]
if not any(p.is_file() for p in fonts):
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-qq','-y','fonts-noto-cjk'],check=True)
print('CJK_FONT_READY',any(p.is_file() for p in fonts),flush=True)


In [ ]:
from pathlib import Path
import sys, zipfile, shutil, hashlib, json, subprocess
from google.colab import files
prefix='PAI_REAL17_COARSE_CAMERA_V1_20261030'
exact=Path('/content/'+prefix+'.zip')
matches=sorted(Path('/content').glob(prefix+'*.zip'))
if exact.is_file(): src=exact
elif matches: src=matches[-1]
else:
    print('ZIPを選択：'+prefix+'.zip',flush=True)
    uploaded=files.upload()
    assert len(uploaded)==1,'1つのZIPだけを選択してください'
    name=next(iter(uploaded))
    assert name.startswith(prefix) and name.endswith('.zip'), '違うZIPが選択された'
    src=Path('/content')/name
root=Path('/content/PAI_REAL17_COARSE_CAMERA_V1_STAGE88')
if root.exists(): shutil.rmtree(root)
root.mkdir(parents=True)
with zipfile.ZipFile(src) as z:
    assert z.testzip() is None,'ZIP破損'
    for n in z.namelist():
        p=Path(n)
        assert not p.is_absolute() and '..' not in p.parts,'不正なZIPパス'
    z.extractall(root)
raw=root/'REAL17_USER_SOURCE_17.zip'
m=json.loads((root/'realphoto/asset_swap_manifest.json').read_text('utf8'))
expected=m['original_photo_sha256']
with zipfile.ZipFile(raw) as z:
    assert z.testzip() is None and set(z.namelist())==set(expected)
    for folder in (root/'REAL17_SOURCE_ORIGINAL_17',root/'realphoto/original_images'):
        folder.mkdir(parents=True,exist_ok=True)
    for n in z.namelist():
        p=Path(n)
        assert len(p.parts)==1 and '..' not in p.parts
        data=z.read(n)
        assert hashlib.sha256(data).hexdigest()==expected[n]
        for folder in (root/'REAL17_SOURCE_ORIGINAL_17',root/'realphoto/original_images'):
            (folder/n).write_bytes(data)
subprocess.run([sys.executable,str(root/'scripts/check_direct_source.py')],cwd=root,check=True)
print('COARSE_SOURCE_READY',root,'ZIP_BYTES',src.stat().st_size,flush=True)


In [ ]:
from pathlib import Path
import subprocess,sys
from google.colab import files
cmd=[sys.executable,'-u',str(root/'run_realphoto_full10.py'),'--repo',str(root),'--out','/content/PAI_REAL17_COARSE_CAMERA_RESULTS','--install']
rc=subprocess.run(cmd,cwd=root,check=False).returncode
print('COARSE_CAMERA_FULL10_RETURN',rc,'0=10/10合格,2=実験未達,3=環境/検査エラー',flush=True)
result=Path('/content/PAI_G8_REAL17_FULL10_RESULT.zip')
if result.is_file():
    print('RESULT_READY',result,result.stat().st_size,flush=True)
    files.download(str(result))
else:
    print('RESULT_ZIP_NOT_FOUND',flush=True)
